<a href="https://colab.research.google.com/github/climatescience/Spatial-data-visualization-with-Python/blob/main/LC_change_map.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import rasterio
import numpy as np
import matplotlib.pyplot as plt


# Input files
lulc_1980 = "/content/LULC_1976.tif"
lulc_2026 = "/content/LULC_2026.tif"


# Read rasters
with rasterio.open(lulc_1980) as src:
    img1980 = src.read(1)

with rasterio.open(lulc_2026) as src:
    img2026 = src.read(1)


# Built-up class value
builtup_class = 2


# Change detection
urban_growth = np.where(
    (img2026 == builtup_class) &
    (img1980 != builtup_class),
    1,
    0
)


# Display
plt.figure(figsize=(10,8))

plt.imshow(
    urban_growth,
    cmap="Reds"
)

plt.title("Built-up Expansion (1980-2026)")
plt.axis("off")

plt.show()



In [ ]:
import rasterio
import numpy as np
import matplotlib.pyplot as plt
import geopandas as gpd


# Input files
lulc_1980 = "/content/LULC_1976.tif"
lulc_2026 = "/content/LULC_2026.tif"

# Shapefile
shapefile = "/content/Faisalabad.shp"


# Read rasters
with rasterio.open(lulc_1980) as src:
    img1980 = src.read(1)
    transform = src.transform
    crs = src.crs

with rasterio.open(lulc_2026) as src:
    img2026 = src.read(1)


# Built-up class value
builtup_class = 2


# Change detection
urban_growth = np.where(
    (img2026 == builtup_class) &
    (img1980 != builtup_class),
    1,
    0
)


# Read boundary
boundary = gpd.read_file(shapefile)

# Match projection with raster
if boundary.crs != crs:
    boundary = boundary.to_crs(crs)

urban_growth_masked = np.ma.masked_where(urban_growth == 0, urban_growth)

# Display
fig, ax = plt.subplots(figsize=(10,8))

plt.imshow(
    urban_growth_masked,
    cmap="Reds",
    vmin=0,
    vmax=1,

    extent=[
        transform[2],
        transform[2] + transform[0]*urban_growth.shape[1],
        transform[5] + transform[4]*urban_growth.shape[0],
        transform[5]
    ]
)


# Add Faisalabad boundary
boundary.boundary.plot(
    ax=ax,
    color="black",
    linewidth=2
)


plt.title("Built-up Expansion (1976-2026)")
plt.axis("off")

plt.show()